# Analyse des anomalies du rapprochement DVF / Cadastre

Objectif : comprendre les parcelles DVF qui ne correspondent pas
exactement à une parcelle cadastrale du Val-de-Marne.

Les anomalies sont réparties en deux catégories :

- `SECTION_MISMATCH` : le numéro de parcelle existe dans la commune,
  mais sous une ou plusieurs sections différentes.
- `NOT_FOUND` : le numéro de parcelle n'est pas retrouvé dans la commune.

In [ ]:
import pandas as pd
import geopandas as gpd

from pathlib import Path

: 

In [ ]:
# Charger les résultats du rapprochement
parcelles = pd.read_csv(
    "../data/parcelles_dvf.csv"
)

# Charger le cadastre
cadastre = gpd.read_file(
    "../data/cadastre/cadastre_94_val_de_marne.geojson"
)

print("Parcelles DVF :", len(parcelles))
print("Parcelles cadastrales :", len(cadastre))

In [ ]:
# Vérifier la répartition des statuts

parcelles["statut_matching"].value_counts()

In [ ]:
# Pourcentage de chaque statut

(
    parcelles["statut_matching"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Parcelles dont le numéro existe dans la commune
# mais sous une autre section

section_mismatch = parcelles[
    parcelles["statut_matching"] == "SECTION_MISMATCH"
].copy()

section_mismatch.head(20)

In [ ]:
# Parcelles totalement absentes de la commune

not_found = parcelles[
    parcelles["statut_matching"] == "NOT_FOUND"
].copy()

not_found.head(20)

In [ ]:
# Préparer une recherche rapide des sections
# associées à chaque couple (commune, numéro)

sections_par_numero = (
    cadastre
    .groupby(
        [
            "commune_normalisee",
            "numero_normalise"
        ]
    )["section_normalisee"]
    .agg(lambda x: sorted(x.dropna().unique()))
    .reset_index()
)

sections_par_numero.head()

In [ ]:
# Ajouter les sections réellement présentes dans le cadastre

section_mismatch = section_mismatch.merge(
    sections_par_numero,
    left_on=[
        "commune_code",
        "numero_normalise"
    ],
    right_on=[
        "commune_normalisee",
        "numero_normalise"
    ],
    how="left"
)

section_mismatch[
    [
        "commune_code",
        "section_normalisee",
        "numero_normalise",
        "section_normalisee_y"
    ]
].head(20)

In [ ]:
# Ajouter les sections réellement présentes dans le cadastre

section_mismatch = section_mismatch.merge(
    sections_par_numero,
    left_on=[
        "commune_code",
        "numero_normalise"
    ],
    right_on=[
        "commune_normalisee",
        "numero_normalise"
    ],
    how="left"
)

section_mismatch[
    [
        "commune_code",
        "section_normalisee",
        "numero_normalise",
        "section_normalisee_y"
    ]
].head(20)

In [ ]:
# Exemple concret d'une anomalie

section_mismatch[
    section_mismatch["commune_code"] == "94077"
].head(10)

In [ ]:
## Conclusion

Le rapprochement  repose sur la clé :

`commune + section + numéro de parcelle`

La majorité des parcelles DVF sont retrouvées directement dans le
cadastre.

Les principales anomalies restantes correspondent à des numéros de
parcelles retrouvés dans la même commune mais associés à une autre
section dans le jeu cadastral.

Ces observations permettent d'identifier les limites du rapprochement
par clé textuelle exacte sans supposer l'origine historique de ces
différences.